In [12]:
import pandas as pd
import numpy as np

# Load the dataset
# Replace 'your_dataset.csv' with the path to your dataset
file_path = '../dataset/Improved_All_Combined_hr_rsp_binary.csv'
data = pd.read_csv(file_path)

# Ensure the data is sorted by Participant and Time
data = data.sort_values(by=['Participant', 'Time(sec)']).reset_index(drop=True)

# Initialize a list to hold the processed windows
windows = []

# Process each participant's data
for participant in data['Participant'].unique():
    participant_data = data[data['Participant'] == participant]
    participant_data = participant_data.reset_index(drop=True)

    # Identify contiguous blocks of the same label
    participant_data['Block'] = (participant_data['Label'] != participant_data['Label'].shift()).cumsum()

    # Process each block separately
    for block_id, block_data in participant_data.groupby('Block'):
        block_label = block_data['Label'].iloc[0]  # Get the label for this block

        # Generate sliding windows within the block
        start_idx = 0
        end_idx = len(block_data) - 1

        while start_idx + 59 <= end_idx:
            # Select a 60-second window
            window_data = block_data.iloc[start_idx:start_idx + 60]

            # Store the window and its label
            windows.append({
                'Participant': participant,
                'Start_Time': window_data['Time(sec)'].iloc[0],
                'End_Time': window_data['Time(sec)'].iloc[-1],
                'HR': window_data['HR'].values.tolist(),
                'Respr': window_data['respr'].values.tolist(),
                'Label': block_label
            })

            # Increment index for sliding window
            start_idx += 1

# Convert the windows list into a DataFrame
windows_df = pd.DataFrame(windows)

# Save the processed dataset to a file (optional)
windows_df.to_csv('processed_windows.csv', index=False)

# Display the first few rows of the processed DataFrame
windows_df.head()

,Participant,Start_Time,End_Time,HR,Respr,Label
0,2,1644227583,1644227642,"[118.0, 113.5, 93.0, 93.25, 86.4, 81.83, 79.71...","[12.1276927, 12.1276927, 12.1276927, 12.127692...",0
1,2,1644227584,1644227643,"[113.5, 93.0, 93.25, 86.4, 81.83, 79.71, 78.12...","[12.1276927, 12.1276927, 12.1276927, 12.127692...",0
2,2,1644227585,1644227644,"[93.0, 93.25, 86.4, 81.83, 79.71, 78.12, 76.67...","[12.1276927, 12.1276927, 12.1276927, 12.127692...",0
3,2,1644227586,1644227645,"[93.25, 86.4, 81.83, 79.71, 78.12, 76.67, 75.6...","[12.1276927, 12.1276927, 12.1276927, 12.127692...",0
4,2,1644227587,1644227646,"[86.4, 81.83, 79.71, 78.12, 76.67, 75.6, 74.82...","[12.1276927, 12.1276927, 12.1276927, 12.127692...",0


In [17]:
from keras.src.callbacks import EarlyStopping
from keras.src.optimizers import Adam
from keras.src.layers import LSTM, Dense
from keras import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Handle missing values
if data.isnull().sum().sum() > 0:
    data = data.dropna()

# Normalize HR and Respr within each participant
scaler = StandardScaler()
data[['HR', 'respr']] = scaler.fit_transform(data[['HR', 'respr']])

# Prepare data for LSTM
X = windows_df[['HR', 'Respr']].apply(lambda row: np.array([row['HR'], row['Respr']]).T, axis=1)
X = np.stack(X.values)
Y = windows_df['Label'].values

# Check for class balance
print(f"Class distribution: {np.bincount(Y)}")

# Split the data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

# Build the LSTM model
model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.001, clipvalue=1.0),
              loss='binary_crossentropy', metrics=['accuracy'])

# Train the model with EarlyStopping to monitor NaN validation loss
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(X_train, Y_train, validation_split=0.2, epochs=20, batch_size=32, verbose=1,
                    callbacks=[early_stopping])

# Evaluate the model
loss, accuracy = model.evaluate(X_test, Y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.2f}")

Class distribution: [67677 30797]
Epoch 1/20


/Users/cedric/miniconda3/envs/project_2/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1970/1970 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.6854 - loss: nan - val_accuracy: 0.6832 - val_loss: nan
Epoch 2/20
1970/1970 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.6874 - loss: nan - val_accuracy: 0.6832 - val_loss: nan
Epoch 3/20
1970/1970 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - accuracy: 0.6868 - loss: nan - val_accuracy: 0.6832 - val_loss: nan
Epoch 4/20
1970/1970 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.6928 - loss: nan - val_accuracy: 0.6832 - val_loss: nan
Epoch 5/20
1970/1970 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.6884 - loss: nan - val_accuracy: 0.6832 - val_loss: nan
Epoch 6/20
1970/1970 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.6872 - loss: nan - val_accuracy: 0.6832 - val_loss: nan
Test Accuracy: 0.69
